In [ ]:
# !mkdir src
# !pip install -q evaluate rouge_score nltk

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
model_checkpoint = "google/mt5-small"
dataset_dict = dict(path = "buruzaemon/amazon_reviews_multi")
min_review_title_length = 3
max_input_length = 256
max_target_length = 30
push_to_hub=False
train_bs = 16
eval_bs = 16
gacc_steps = 2
num_train_epochs = 3
lr=2e-5
eval_on_start = True
lr_scheduler_type = "linear"
task = "summarization"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path'].split("/")[-1]}-accelerate"

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

In [ ]:
%%writefile src/data.py


from datasets import load_dataset
from datasets import concatenate_datasets, DatasetDict
from src.config import dataset_dict, min_review_title_length

def filter_books(examples):
    return [(pcat == "book") for pcat in examples["product_category"]]
    

def load_datasets():
    lang = "en"
    url = f"https://huggingface.co/datasets/{dataset_dict["path"]}/resolve/main/{lang}"
    data_files = {
        split: f"{url}/{split}.jsonl.gz" for split in ["train", "test", "validation"]
    }
    english_dataset = load_dataset("json", data_files=data_files)
    
    
    lang = "es"
    url = f"https://huggingface.co/datasets/{dataset_dict["path"]}/resolve/main/{lang}"
    data_files = {
        split: f"{url}/{split}.jsonl.gz" for split in ["train", "test", "validation"]
    }
    spanish_dataset = load_dataset("json", data_files=data_files)

    return english_dataset, spanish_dataset

def prep_datasets(english_dataset, spanish_dataset):
    spanish_books = spanish_dataset.filter(filter_books, batched=True)
    english_books = english_dataset.filter(filter_books, batched=True)
    
    books_dataset = DatasetDict()
    for split in english_books.keys():
        books_dataset[split] = concatenate_datasets(
            [english_books[split], spanish_books[split]]
        )
        books_dataset[split] = books_dataset[split].shuffle(seed=42)
    
    books_dataset = books_dataset.filter(
        lambda x: map(lambda x: len(x.split()) >= min_review_title_length, x["review_title"]),
        batched=True
    )
    return books_dataset

In [ ]:
%%writefile src/preprocess.py

from src.config import max_input_length, max_target_length

def preprocess_function(examples, tokenizer):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        text_target = examples["review_title"],
        max_length=max_target_length,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
%%writefile src/evaluation.py

from nltk.tokenize import sent_tokenize
import numpy as np
import torch, evaluate
from tqdm.auto import tqdm
from src.config import max_target_length, min_review_title_length

def postprocess(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # ROUGE expects a newline after each sentence
    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])

def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])



def run_eval(model, eval_dataloader, accelerator, tokenizer):
    # TODO: test how this works | why don't we pass max length to generate here???
    # TODO: compare this to the translation run_eval and finalize it
    model.eval()
    rouge_score = evaluate.load("rouge")
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length = max_target_length,
                min_length = min_review_title_length
            )

            pad_id = tokenizer.pad_token_id
            generated_tokens = accelerator.pad_across_processes(generated_tokens,dim=1,pad_index=pad_id)
            # Since we did not pad to max length, we need to pad the labels too
            labels = accelerator.pad_across_processes(batch["labels"],dim=1,pad_index=pad_id)

            generated_tokens = accelerator.gather(generated_tokens).cpu().numpy()
            labels = accelerator.gather(labels).cpu().numpy()

            # Replace -100 in the labels as we can't decode them
            labels = np.where(labels != -100, labels, pad_id)
            if isinstance(generated_tokens, tuple): generated_tokens = generated_tokens[0]
            decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
            decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
            decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)

            rouge_score.add_batch(predictions=decoded_preds, references=decoded_labels)

    result = rouge_score.compute()
    result = {key: round(value * 100, 4) for key, value in result.items()}
    return result

In [ ]:
%%writefile src/train.py

from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from src.config import train_bs, eval_bs
from src.preprocess import preprocess_function

def create_dls(books_dataset, tokenizer, model):
    tokenized_datasets = books_dataset.map(
        preprocess_function,
        batched=True,
        fn_kwargs = {"tokenizer":tokenizer},
        remove_columns=books_dataset["train"].column_names,
    )
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
    
    train_dataloader = DataLoader(
        tokenized_datasets["train"],
        collate_fn=data_collator,
        batch_size=train_bs,
        shuffle=True,
    )
    eval_dataloader = DataLoader(
        tokenized_datasets["validation"],
        collate_fn=data_collator,
        batch_size=eval_bs
    )
    return train_dataloader, eval_dataloader

In [ ]:
# import os, torch, evaluate, nltk
# from transformers import AutoTokenizer
# from transformers import AutoModelForSeq2SeqLM
# from src.config import *
# from src.utils import hf_login, get_mp
# from src.data import load_datasets, prep_datasets
# from src.preprocess import preprocess_function
# from src.train import create_dls

# nltk.download("punkt")

# hf_login()
# english_dataset, spanish_dataset = load_datasets()
# books_dataset = prep_datasets(english_dataset, spanish_dataset)
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
# model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
# train_dataloader, eval_dataloader = create_dls(books_dataset, tokenizer, model)

In [ ]:
# from src.evaluation import postprocess
# import numpy as np

# batch = next(iter(eval_dataloader))
# print(batch["input_ids"].shape)
# with torch.no_grad():
#     generated_tokens = model.generate(
#         batch["input_ids"],
        # attention_mask=batch["attention_mask"],
        # max_length = max_target_length,
        # min_length = min_review_title_length
#     )
#     if isinstance(generated_tokens, tuple): generated_tokens = generated_tokens[0]

#     labels = batch["labels"]
#     # Replace -100 in the labels as we can't decode them
#     pad_id = tokenizer.pad_token_id
#     labels = np.where(labels != -100, labels, pad_id)
#     decoded_preds = tokenizer.batch_decode(
#         generated_tokens, skip_special_tokens=True
#     )
#     decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
#     decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)

# for p,l in zip(decoded_preds, decoded_labels):
#     print(f"label: {l} <|> prediction: {p}")

In [ ]:
# import evaluate
# from src.evaluation import evaluate_baseline 
# rouge_score = evaluate.load("rouge")
# score = evaluate_baseline(books_dataset["validation"], rouge_score)
# rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
# rouge_dict = dict((rn, round(score[rn] * 100, 2)) for rn in rouge_names)
# print(rouge_dict)

In [ ]:
%%writefile main.py

import numpy as np
import os, torch, evaluate, nltk,math
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_scheduler
from huggingface_hub import HfApi, create_repo
from torch.optim import AdamW
from accelerate import Accelerator
from accelerate.utils import DistributedDataParallelKwargs

from src.config import *
from src.utils import hf_login, get_mp
from src.data import load_datasets, prep_datasets
from src.train import create_dls
from src.evaluation import run_eval

nltk.download("punkt")

if __name__ == "__main__":
    # Init accelerator
    ddp_kwargs = DistributedDataParallelKwargs(
        find_unused_parameters=True
    )
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp(),
        kwargs_handlers=[ddp_kwargs],
    )
    
    if accelerator.is_main_process:
        accelerator.print(f"checkpoint name: {ckpt_name}")
        accelerator.print(f"# GPUs: {accelerator.num_processes}")
        effective_bs = train_bs * gacc_steps * accelerator.num_processes
        accelerator.print(f"effective_batch_size: {effective_bs}")
    # HF Login and repo initiation
    if accelerator.is_main_process:
        hf_login()
        hf_api = HfApi()
        accelerator.print(f"HF user: {hf_api.whoami()['name']}")
        if push_to_hub:
            # Create the HF repo to push the model to the HF hub
            HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
            create_repo(repo_id=HF_REPO_ID, exist_ok=True)
            accelerator.print(f"HF repo: {HF_REPO_ID}")

    with accelerator.main_process_first():
        # Load model, tokenizer, and create dataloaders
        english_dataset, spanish_dataset = load_datasets()
        books_dataset = prep_datasets(english_dataset, spanish_dataset)
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
        model.config.use_cache = False
        train_dataloader, eval_dataloader = create_dls(books_dataset, tokenizer, model)

    if accelerator.is_main_process:
        train_ds = train_dataloader.dataset
        eval_ds = eval_dataloader.dataset
        accelerator.print(f"final input datasets features: {train_ds.features}")
        accelerator.print(f"train DS length: {len(train_ds)}")
        accelerator.print(f"eval DS length: {len(eval_ds)}")

    # Preparing training components
    optimizer = AdamW(model.parameters(), lr=lr)
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = max(1, num_update_steps_per_epoch // 4)
    eval_steps = num_update_steps_per_epoch # eval with `generate()` is expensive
    
    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    pid = f"[rank={rank} main={is_main}]"
    accelerator.print(f"{pid}|train_len={len(train_dataloader)}|eval_len={len(eval_dataloader)}")
    accelerator.print(f"{pid}|eval_steps={eval_steps}|train_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone()


    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
        leave=False,
    )

    if eval_on_start:
        eval_logs = run_eval(model, eval_dataloader, accelerator, tokenizer)
        if accelerator.is_main_process:
            accelerator.print(f"Step {global_step} eval:", eval_logs)

    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if accelerator.sync_gradients:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    loss_all = accelerator.gather(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather(lr_t).detach().cpu().tolist()
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            msg = f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                            accelerator.print(msg)
                    running_loss = 0.0
                    running_mb = 0
                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(model, eval_dataloader, accelerator, tokenizer)
                    msg = f"Step {global_step} eval:", eval_logs
                    if accelerator.is_main_process: accelerator.print(msg)


    if accelerator.is_main_process:
        accelerator.print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training()
    accelerator.free_memory()

In [ ]:
! accelerate launch --num_processes 2 main.py